# secmcp en GPU gratuita (Colab o Kaggle)
Sirve un modelo abierto con transformers en una GPU T4, le mide latencia y rendimiento, y corre el agente con y sin Guard.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → GPU T4.
Los resultados dependen de la GPU que te toque y de la carga de la sesión: pegá la salida tal cual en el README, sin editarla.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('CUDA:', torch.cuda.is_available())

## 1. Traer el repositorio
Usá el clon (cambiá la URL) o subí el zip al panel de archivos.

In [ ]:
import os
REPO_URL = "https://github.com/fleonmoloney/centro-ia-secure-mcp-gpu"  # cambiar si el repo tiene otro nombre
if not os.path.exists("secmcp-repo"):
    !git clone $REPO_URL secmcp-repo || echo "Clon falló: subí el zip y descomprimilo en secmcp-repo"
%cd secmcp-repo
!pip install -q -e ".[gpu,dev]"

## 2. Pruebas y compuerta de evals (no usan GPU)

In [ ]:
!python -m pytest -q
!python -m secmcp.evals --gate

## 3. Levantar el servicio de modelos con GPU
Primera vez: descarga el modelo (~3 GB para 1.5B en fp16).

In [ ]:
import subprocess, time, httpx, os, signal

def start(quant="none", model="Qwen/Qwen2.5-1.5B-Instruct", port=8000):
    stop()
    env = dict(os.environ, BACKEND="transformers", MODEL_ID=model, QUANT=quant, SERVE_API_KEY="demo")
    global proc
    proc = subprocess.Popen(["uvicorn", "secmcp.serving.app:app_factory", "--factory", "--port", str(port)],
                            env=env, stdout=open("serving.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(300):
        try:
            r = httpx.get(f"http://127.0.0.1:{port}/readyz")
            if r.status_code == 200:
                return r.json()
        except httpx.HTTPError:
            pass
        time.sleep(2)
    raise RuntimeError("no levantó; mirá serving.log")

def stop():
    global proc
    if "proc" in globals() and proc.poll() is None:
        proc.send_signal(signal.SIGTERM); proc.wait(30)

print(start("none"))

## 4. Benchmark
Una corrida en esta GPU, con 20 pedidos. No es una garantía ni un benchmark de referencia.

In [ ]:
!python -m secmcp.serving.bench --key demo -n 20 -c 1 --max-tokens 64
!python -m secmcp.serving.bench --key demo -n 20 -c 4 --max-tokens 64
print(httpx.get("http://127.0.0.1:8000/v1/info", headers={"Authorization": "Bearer demo"}).json())
# si hubo errores, el motivo real está en el log del servidor:
!tail -n 40 serving.log


## 5. Comparar fp16 contra 4 bits (bitsandbytes)
Cambia memoria de GPU y latencia; la calidad en tu tarea hay que medirla aparte.

In [ ]:
print(start("4bit"))
!python -m secmcp.serving.bench --key demo -n 20 -c 1 --max-tokens 64
print(httpx.get("http://127.0.0.1:8000/v1/info", headers={"Authorization": "Bearer demo"}).json())
# si hubo errores, el motivo real está en el log del servidor:
!tail -n 40 serving.log


## 6. Agente ante un documento envenenado, con el modelo servido
Un modelo de 1.5B puede no emitir JSON válido o no obedecer la inyección: el resultado depende del modelo y se imprime sin retocar.
La garantía de la política no depende de esto: está en la compuerta determinista del paso 2.

In [ ]:
!python -m secmcp.demo --key demo

## 6b. Calidad: fp16 contra 4 bits
Mide la perplejidad (inglés y español) y el comportamiento como agente (¿emite una acción JSON válida? ¿elige la acción esperada?) con el modelo cargado en el propio proceso. Muestras chicas: son un indicio, no una prueba.
Primero libera la GPU (servicio de transformers y vLLM, si estuvieran activos). Tarda unos minutos.

In [ ]:
stop()
!pkill -f "vllm serve" || true
!python -m secmcp.quality --quant none --out results/q_fp16.json > /dev/null
!python -m secmcp.quality --quant 4bit --out results/q_4bit.json > /dev/null
!python -m secmcp.quality --compare results/q_fp16.json results/q_4bit.json


## 7. (Opcional) Servir con vLLM
vLLM da batching continuo y es lo que corresponde para carga real. La API es compatible con OpenAI, así que el benchmark y la demo de arriba funcionan igual.
Compatibilidad con la T4 y con la versión de torch de Colab: verificarla; puede requerir fijar versiones.

In [ ]:
# Opcional. Antes de lanzar vLLM hay que liberar el puerto 8000 (si no, el benchmark mide el servidor anterior).
stop()
!pip install -q vllm
!nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8000 --dtype half --max-model-len 4096 --api-key demo > vllm.log 2>&1 &
import time, httpx
for _ in range(180):  # vLLM tarda en cargar; se espera a que responda /v1/models
    try:
        if httpx.get("http://127.0.0.1:8000/v1/models", headers={"Authorization": "Bearer demo"}).status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(5)
else:
    raise RuntimeError("vLLM no levantó; mirá vllm.log")
!python -m secmcp.serving.bench --key demo -n 20 -c 8
!tail -n 15 vllm.log


In [ ]:
stop()